# Day 5: zero-shot VLM execution scoring on FineGym vault clips

**Question:** can a general vision-language model (no training on gymnastics) judge *execution* (E) from frames?

**Runtime → Change runtime type → T4 GPU**, then *Runtime → Run all*.

- Model: `Qwen/Qwen2.5-VL-3B-Instruct` (fits a T4 in float16). Fallback in section 3 if its output is garbage.
- Clips: **36 test clips** = 12 from each of the 3 most common vault codes, spread across competitions (so within-vault comparison is possible).
- Input: 8 frames per clip in time order (3 approach, 2 flight, 3 landing, using the stage timestamps), plus the vault's difficulty value.
- Output: one short phrase per stage + an E score. Saved after **every clip** to `capstone/vlm/vlm_scores.csv`, so a disconnect loses nothing: just *Run all* again.

Expected time: ~5 min setup + ~10–20 min for 36 clips.

Evaluation is done afterwards on your laptop with `analysis/day5_eval_vlm.py`.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
!pip -q install -U "transformers>=4.49" accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 97.4 MB/s eta 0:00:00


## 1. Paths and clip subset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd, numpy as np, shutil
ROOT   = Path('/content/drive/MyDrive/capstone')
CLIPS  = ROOT / 'clips'
OUTDIR = ROOT / 'vlm'; OUTDIR.mkdir(exist_ok=True)
LOCAL  = Path('/content/clips'); LOCAL.mkdir(exist_ok=True)
OUT    = OUTDIR / 'vlm_scores.csv'

N_PER_CODE, N_CODES, SEED = 12, 3, 42

df = pd.read_csv(ROOT / 'out' /'clips_available.csv', dtype={'vault_code': str, 'bib': str})
df['clip_id'] = df.video_id + '__' + df.event_key
test = df[(df.split == 'test') & df.segment_key.notna()]
codes = test.vault_code.value_counts().index[:N_CODES].tolist()

# round-robin over videos inside each vault code, so the subset spans several competitions
picked = []
for c in codes:
    g = test[test.vault_code == c].sample(frac=1, random_state=SEED)
    by_vid = [v for _, v in g.groupby('video_id')]
    i = 0
    while sum(len(p) for p in picked if p.vault_code.iloc[0] == c) < N_PER_CODE and any(i < len(v) for v in by_vid):
        for v in by_vid:
            if i < len(v) and sum(len(p) for p in picked if p.vault_code.iloc[0] == c) < N_PER_CODE:
                picked.append(v.iloc[[i]])
        i += 1
subset = pd.concat(picked).reset_index(drop=True)
subset.to_csv(OUTDIR / 'vlm_subset.csv', index=False)
print('vault codes:', codes)
print(subset.groupby('vault_code').agg(clips=('clip_id', 'size'), videos=('video_id', 'nunique'),
                                       E_min=('E', 'min'), E_max=('E', 'max')))

for f in subset.clip_file:
    if not (LOCAL / f).exists():
        shutil.copy(CLIPS / f, LOCAL / f)
print('local clips ready:', len(subset))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
vault codes: ['5.80', '5.40', '4.60']
            clips  videos  E_min  E_max
vault_code                             
4.60           12       6  8.600  8.900
5.40           12       7  8.600  9.233
5.80           12      12  8.633  9.400
local clips ready: 36


## 2. Frame sampling (same functions as Day 3)

In [3]:
import json
import math

import cv2
import numpy as np

N_FRAMES = 16  # VideoMAE input length


def read_clip(path, short_side=256):
    """Read every frame of a short clip into memory (RGB, resized so the short side = short_side)."""
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    frames = []
    while True:
        ok, f = cap.read()
        if not ok:
            break
        h, w = f.shape[:2]
        s = short_side / min(h, w)
        f = cv2.resize(f, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames, fps


def sample_window(frames, fps, t0, t1, n=N_FRAMES):
    """n frames evenly spaced in [t0, t1) seconds (centre of n equal bins); clamps to the clip."""
    last = len(frames) - 1
    t1 = max(t1, t0 + 1e-3)
    times = [t0 + (i + 0.5) * (t1 - t0) / n for i in range(n)]
    idx = [min(last, max(0, int(math.floor(t * fps)))) for t in times]
    return [frames[i] for i in idx], idx


def windows_for_row(row):
    """(whole, [stage1, stage2, stage3] or None) windows in seconds inside the downloaded file.
    Annotation times are relative to the vault start; the file starts `stage_offset` s earlier."""
    off = float(row["stage_offset"])
    whole = (off, off + float(row["duration"]))
    stages = None
    raw = row.get("stages")
    if isinstance(raw, str) and raw.strip().startswith("["):
        st = json.loads(raw)
        if len(st) == 3:
            stages = [(off + a, off + b) for a, b in st]
    return whole, stages


from PIL import Image

def frames_for_vlm(row, short_side=336):
    """8 PIL frames in time order: 3 approach, 2 flight, 3 landing (or 8 uniform if no stages)."""
    fr, fps = read_clip(LOCAL / row['clip_file'], short_side=short_side)
    whole, stages = windows_for_row(row)
    if stages:
        picks = []
        for (a, b), n in zip(stages, [3, 2, 3]):
            picks += sample_window(fr, fps, a, b, n=n)[0]
    else:
        picks = sample_window(fr, fps, *whole, n=8)[0]
    return [Image.fromarray(f) for f in picks]

## 3. Model

If the smoke test below prints empty text, repeated symbols, or no `E:` line, Qwen2.5-VL is misbehaving in float16 on this GPU. Then set `MODEL_ID = 'Qwen/Qwen2-VL-2B-Instruct'` and `DTYPE = torch.float32` in this cell, *Runtime → Restart session*, and *Run all* again (already-scored clips are skipped; note the model change in your notes).

In [4]:
import torch, re
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
DTYPE = torch.float16

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map='cuda').eval()

PROMPT = '''You are an experienced FIG judge for women's artistic gymnastics.
The {n} images are frames of ONE vault, in time order: frames 1-3 approach and take-off, frames 4-5 flight, frames 6-8 landing.
The vault's difficulty value is {D}.
Judge only EXECUTION (the E score, out of 10.0; at this competition level E is usually between 7.5 and 9.8).
Consider: body shape and leg form in flight, height and distance, and the landing (steps, hops, balance, chest position).
Reply in exactly this format and nothing else:
Approach: <short phrase>
Flight: <short phrase>
Landing: <short phrase>
E: <score with one decimal>'''

def parse_E(text):
    m = re.findall(r'E\s*[:=]\s*([0-9]+(?:\.[0-9]+)?)', text)
    if not m:
        return np.nan
    v = float(m[-1])
    return v if 0 <= v <= 10 else np.nan

@torch.no_grad()
def score_clip(row):
    imgs = frames_for_vlm(row)
    content = [{'type': 'image'} for _ in imgs] + [{'type': 'text', 'text': PROMPT.format(n=len(imgs), D=row['D'])}]
    msgs = [{'role': 'user', 'content': content}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=imgs, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
    return processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0].strip()

# smoke test
r = subset.iloc[0].to_dict()
resp = score_clip(r)
print(resp, '\n-> parsed E:', parse_E(resp), '| true E:', r['E'])

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Approach: Smooth and confident
Flight: Controlled and fluid
Landing: Stable and balanced
E: 8.2 
-> parsed E: 8.2 | true E: 9.133


## 4. Score all clips (resumable, saved after each clip)

In [5]:
import time
from tqdm.auto import tqdm

done = set(pd.read_csv(OUT).clip_id) if OUT.exists() else set()
todo = subset[~subset.clip_id.isin(done)]
print(f'{len(done)} done, {len(todo)} to go')

t0 = time.time()
for _, row in tqdm(todo.iterrows(), total=len(todo)):
    r = row.to_dict()
    try:
        resp = score_clip(r); err = ''
    except Exception as e:
        resp, err = '', repr(e)[:200]
    rec = pd.DataFrame([{'clip_id': r['clip_id'], 'video_id': r['video_id'], 'vault_code': r['vault_code'],
                         'D': r['D'], 'E_true': r['E'], 'E_vlm': parse_E(resp), 'response': resp,
                         'error': err, 'model': MODEL_ID}])
    rec.to_csv(OUT, mode='a', header=not OUT.exists(), index=False)
print(f'done in {(time.time() - t0) / 60:.1f} min')

0 done, 36 to go


  0%|          | 0/36 [00:00<?, ?it/s]

done in 3.5 min


## 5. Quick look (the real evaluation runs on your laptop)

In [6]:
res = pd.read_csv(OUT)
print(len(res), 'clips | parsed:', res.E_vlm.notna().sum(), '| errors:', (res.error.fillna('') != '').sum())
print('VLM E: mean %.2f  std %.3f  distinct values %d' % (res.E_vlm.mean(), res.E_vlm.std(), res.E_vlm.nunique()))
print('true E: mean %.2f  std %.3f' % (res.E_true.mean(), res.E_true.std()))
for t in res.response.head(3): print('---\n' + str(t))

36 clips | parsed: 33 | errors: 0
VLM E: mean 8.20  std 0.000  distinct values 1
true E: mean 8.92  std 0.214
---
Approach: Smooth and confident
Flight: Controlled and fluid
Landing: Stable and balanced
E: 8.2
---
Approach: Smooth and confident
Flight: Graceful and controlled
Landing: Stable and balanced
E: 8.2
---
Approach: Smooth and steady
Flight: Controlled and fluid
Landing: Stable and balanced
E: 8.2


## 6. Optional pairwise VLM comparison

Compare two athletes performing the same vault. The VLM must decide which
performance has better execution, rather than assigning an absolute E score.

Pairs are selected from the same vault code but different source videos,
with a minimum E-score difference of 0.2.

In [8]:
# Build same-vault, different-video comparison pairs.
# We use the existing 36-clip subset and its ground-truth E scores.

PAIR_GAP = 0.2
MAX_PAIRS = 24
PAIR_SEED = 123

rng_pair = np.random.default_rng(PAIR_SEED)

pairs = []

for vault in subset.vault_code.unique():
    g = subset[subset.vault_code == vault].reset_index(drop=True)

    candidates = []

    for i in range(len(g)):
        for j in range(i + 1, len(g)):
            # Same vault, different source videos
            if g.loc[i, 'video_id'] == g.loc[j, 'video_id']:
                continue

            # Require a meaningful execution-score difference
            gap = abs(float(g.loc[i, 'E']) - float(g.loc[j, 'E']))
            if gap < PAIR_GAP:
                continue

            # A = first clip, B = second clip
            # Ground-truth winner is determined by E.
            winner = 'A' if g.loc[i, 'E'] > g.loc[j, 'E'] else 'B'

            candidates.append({
                'vault_code': vault,
                'clip_A': g.loc[i, 'clip_id'],
                'clip_B': g.loc[j, 'clip_id'],
                'video_A': g.loc[i, 'video_id'],
                'video_B': g.loc[j, 'video_id'],
                'E_A': float(g.loc[i, 'E']),
                'E_B': float(g.loc[j, 'E']),
                'true_winner': winner,
                'gap': gap,
            })

    # Shuffle candidates within this vault
    rng_pair.shuffle(candidates)
    pairs.extend(candidates)

# Shuffle across vaults and keep a fixed number
rng_pair.shuffle(pairs)
pairs = pairs[:MAX_PAIRS]

pairs_df = pd.DataFrame(pairs)

print(f"candidate/selected pairs: {len(pairs_df)}")
print(
    pairs_df.groupby('vault_code')
    .size()
    .rename('pairs')
)

print(
    pairs_df[['vault_code', 'E_A', 'E_B', 'gap', 'true_winner']]
    .head(10)
)

pairs_df.to_csv(OUTDIR / 'vlm_pairwise_subset.csv', index=False)

candidate/selected pairs: 24
vault_code
4.60     4
5.40     7
5.80    13
Name: pairs, dtype: int64
  vault_code    E_A    E_B    gap true_winner
0       5.80  9.366  8.658  0.708           A
1       5.40  9.233  8.950  0.283           A
2       5.80  8.633  9.166  0.533           B
3       4.60  8.600  8.866  0.266           B
4       5.40  9.233  8.600  0.633           A
5       4.60  8.600  8.800  0.200           B
6       5.80  9.150  8.833  0.317           A
7       5.80  9.400  8.633  0.767           A
8       5.80  9.133  8.633  0.500           A
9       5.40  8.600  9.166  0.566           B


In [9]:
PAIR_PROMPT = '''You are an experienced FIG judge for women's artistic gymnastics.

You are comparing TWO athletes performing the SAME vault.

You will see 8 frames from Athlete A followed by 8 frames from Athlete B.
For each athlete, the frames are in time order:
- frames 1-3: approach and take-off
- frames 4-5: flight
- frames 6-8: landing

The vault is the same for both athletes.

Judge ONLY execution quality.
Consider body shape and leg form in flight, height and distance,
and especially the landing: steps, hops, balance, and chest position.

Do NOT judge difficulty.
Do NOT use the athletes' identities.
Do NOT assume that A or B is better because of presentation order.

Reply in exactly this format and nothing else:

Winner: A
Reason: <one short sentence>

or

Winner: B
Reason: <one short sentence>
'''

def parse_winner(text):
    m = re.findall(r'Winner\s*:\s*([AB])\b', text, flags=re.IGNORECASE)
    if not m:
        return None
    return m[-1].upper()


@torch.no_grad()
def compare_pair(row):
    a = subset[subset.clip_id == row.clip_A].iloc[0]
    b = subset[subset.clip_id == row.clip_B].iloc[0]

    imgs_A = frames_for_vlm(a)
    imgs_B = frames_for_vlm(b)

    imgs = imgs_A + imgs_B

    content = (
        [{'type': 'image'} for _ in imgs]
        + [{'type': 'text',
            'text': PAIR_PROMPT}]
    )

    msgs = [{'role': 'user', 'content': content}]

    text = processor.apply_chat_template(
        msgs,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=imgs,
        return_tensors='pt'
    ).to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

    response = processor.batch_decode(
        out[:, inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )[0].strip()

    return response

In [10]:
PAIR_OUT = OUTDIR / 'vlm_pairwise_scores.csv'

existing = (
    pd.read_csv(PAIR_OUT)
    if PAIR_OUT.exists()
    else pd.DataFrame()
)

done_pairs = set()

if len(existing):
    done_pairs = set(
        zip(existing.clip_A, existing.clip_B)
    )

todo = pairs_df[
    ~pairs_df.apply(
        lambda r: (r.clip_A, r.clip_B) in done_pairs,
        axis=1
    )
]

print(f'{len(done_pairs)} done, {len(todo)} to go')

for _, row in todo.iterrows():
    try:
        response = compare_pair(row)
        error = ''
        predicted = parse_winner(response)
    except Exception as e:
        response = ''
        error = repr(e)[:200]
        predicted = None

    rec = pd.DataFrame([{
        'vault_code': row.vault_code,
        'clip_A': row.clip_A,
        'clip_B': row.clip_B,
        'video_A': row.video_A,
        'video_B': row.video_B,
        'E_A': row.E_A,
        'E_B': row.E_B,
        'true_winner': row.true_winner,
        'pred_winner': predicted,
        'correct': (
            predicted == row.true_winner
            if predicted is not None else False
        ),
        'gap': row.gap,
        'response': response,
        'error': error,
        'model': MODEL_ID
    }])

    rec.to_csv(
        PAIR_OUT,
        mode='a',
        header=not PAIR_OUT.exists(),
        index=False
    )

print('done')

0 done, 24 to go
done


In [11]:
res_pair = pd.read_csv(PAIR_OUT)

valid = res_pair.pred_winner.notna()

print(
    f"pairs: {len(res_pair)} | "
    f"valid comparisons: {valid.sum()} | "
    f"errors: {(res_pair.error.fillna('') != '').sum()}"
)

if valid.sum():
    accuracy = (
        res_pair.loc[valid, 'pred_winner']
        == res_pair.loc[valid, 'true_winner']
    ).mean()

    print(f"VLM pairwise accuracy: {accuracy:.3f}")
    print("chance level:          0.500")

    print("\nPrediction counts:")
    print(res_pair.loc[valid, 'pred_winner'].value_counts())

    print("\nExamples:")
    for _, r in res_pair[valid].head(5).iterrows():
        print("---")
        print(
            f"Vault {r.vault_code} | "
            f"A E={r.E_A:.3f} | B E={r.E_B:.3f} | "
            f"true={r.true_winner} | predicted={r.pred_winner}"
        )
        print(r.response)

pairs: 24 | valid comparisons: 18 | errors: 0
VLM pairwise accuracy: 0.333
chance level:          0.500

Prediction counts:
pred_winner
B    18
Name: count, dtype: int64

Examples:
---
Vault 5.4 | A E=9.233 | B E=8.950 | true=A | predicted=B
Winner: B
Reason: B has a more stable landing with fewer steps and a better chest position compared to A.
---
Vault 5.8 | A E=8.633 | B E=9.166 | true=B | predicted=B
Winner: B
Reason: B has a more stable landing with fewer steps and a better chest position compared to A.
---
Vault 4.6 | A E=8.600 | B E=8.866 | true=B | predicted=B
Winner: B
Reason: B has a more stable landing with fewer steps and a better chest position compared to A.
---
Vault 5.4 | A E=9.233 | B E=8.600 | true=A | predicted=B
Winner: B
Reason: B has a more stable landing with fewer steps and a better chest position compared to A.
---
Vault 4.6 | A E=8.600 | B E=8.800 | true=B | predicted=B
Winner: B
Reason: B has a more stable landing with fewer steps and a better chest position